### Next Steps:

**The following for the CNN architecture first:**
- make models flexible to differnt input lengths and for amino aids
- create orchestrator to train
    - one model
    - with one specific hyperparameter setting
    - for one dataset using pytorch-lightning (not in container)
- integrate hyperparameter search and think about tracking that using ml-flow

In [1]:
from pathlib import Path

def print_tree(root_dir, prefix=""):
    root = Path(root_dir)
    entries = sorted(root.iterdir(), key=lambda x: (x.is_file(), x.name.lower()))

    for i, entry in enumerate(entries):
        connector = "└── " if i == len(entries) - 1 else "├── "
        print(prefix + connector + entry.name)

        if entry.is_dir():
            extension = "    " if i == len(entries) - 1 else "│   "
            print_tree(entry, prefix + extension)

print_tree('C:/Users/kroep/Desktop/AI/ssl-for-ood-molecules')

├── .git
│   ├── hooks
│   │   ├── applypatch-msg.sample
│   │   ├── commit-msg.sample
│   │   ├── fsmonitor-watchman.sample
│   │   ├── post-update.sample
│   │   ├── pre-applypatch.sample
│   │   ├── pre-commit.sample
│   │   ├── pre-merge-commit.sample
│   │   ├── pre-push.sample
│   │   ├── pre-rebase.sample
│   │   ├── pre-receive.sample
│   │   ├── prepare-commit-msg.sample
│   │   ├── push-to-checkout.sample
│   │   ├── sendemail-validate.sample
│   │   └── update.sample
│   ├── info
│   │   └── exclude
│   ├── logs
│   │   ├── refs
│   │   │   ├── heads
│   │   │   │   └── main
│   │   │   └── remotes
│   │   │       └── origin
│   │   │           └── main
│   │   └── HEAD
│   ├── objects
│   │   ├── 09
│   │   │   └── 9123f24b0b4ebe8dbb24136f13156acf82f98e
│   │   ├── 1c
│   │   │   └── d11c0e08b6dfdff4b65aa7b488d36f5d248292
│   │   ├── 23
│   │   │   └── 5b6b29bddfff301a13893be442e2f8b69cf79f
│   │   ├── 2b
│   │   │   └── 008a2c04f6a8cf3eed39d309dcd41f49f2f3da
│   │   ├── 2d

In [2]:
'src.md'.split('.')

['src', 'md']

### Imports

In [3]:
import torch
if torch.cuda.is_available():
    print(f'Using GPU: {torch.cuda.get_device_name(0)}')
else:
    print('No GPU found')
    
from sklearn.model_selection import train_test_split
from collections import Counter
import matplotlib.pyplot as plt
from matplotlib import gridspec
from matplotlib.lines import Line2D
import os
import re
import numpy as np
from collections import defaultdict
import random
import json
import time

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
import pandas as pd

#from models.benchmarks.cnn_lstm_model import train_cnn_lstm_from_config

#from models.benchmarks.mlp_model import train_mlp_from_config
#from models.benchmarks.cnn_model import train_cnn_from_config
#from models.benchmarks.lstm_model import train_lstm_from_config
#from models.benchmarks.lstm_cnn_model import train_lstm_cnn_from_config
#from models.benchmarks.cnn_lstm_model import train_cnn_lstm_from_config
#from models.benchmarks.transformer_model import train_transformer_from_config
#from functions.evaluate_best_models import run_evaluation
#from functions.evaluate_best_models import predict_y_values

#from functions.random_search import random_search

from huggingface_hub import hf_hub_download

Using GPU: NVIDIA GeForce GTX 1660 SUPER


### Load data

In [4]:
from ssl_for_ood.data.load_data import load_all_data
all_data = load_all_data()

### Preprocess data

In [5]:
from ssl_for_ood.data.preprocess_data import preprocess_all_data
all_prep_data = preprocess_all_data(overwrite=True)

### Load preprocessed datasets

In [6]:
datasets = ['GFP', 'AAV', 'TFBind8']
data_dict_df = {dataset: {} for dataset in datasets}
data_dict_np = {dataset: {} for dataset in datasets}

for split in ['train', 'val_id', 'val_ood', 'test']:
    data_dict_df['GFP'][split] = pd.read_csv(os.path.join('data', 'gfp', 'splits', f'{split}.csv'))
    data_dict_df['AAV'][split] = pd.read_csv(os.path.join('data', 'aav', 'splits', f'{split}.csv'))
    data_dict_df['TFBind8'][split] = pd.read_csv(os.path.join('data', 'tfbind8', 'splits', f'{split}.csv'))

for split in ['train', 'val_id', 'val_ood', 'test']:
    data_dict_np['GFP'][split] = {
        'x': np.load(os.path.join('data', 'gfp', 'preprocessed', f'{split}.npz'))['x'],
        'y': np.load(os.path.join('data', 'gfp', 'preprocessed', f'{split}.npz'))['y']
    }
    data_dict_np['AAV'][split] = {
        'x': np.load(os.path.join('data', 'aav', 'preprocessed', f'{split}.npz'))['x'],
        'y': np.load(os.path.join('data', 'aav', 'preprocessed', f'{split}.npz'))['y']
    }
    data_dict_np['TFBind8'][split] = {
        'x': np.load(os.path.join('data', 'tfbind8', 'preprocessed', f'{split}.npz'))['x'],
        'y': np.load(os.path.join('data', 'tfbind8', 'preprocessed', f'{split}.npz'))['y']
    }

### Train Models

**Think about:**
- saving results
- ML Flow

In [7]:
import numpy as np
from ssl_for_ood.data.preprocess_data import load_y_scaler, transform_y, inverse_transform_y

scaler = load_y_scaler("aav")

y = np.array([-1, 0, 1, 2, 5], dtype=np.float32)

y_scaled = transform_y(y, scaler)
y_back = inverse_transform_y(y_scaled, scaler)

### ML-Flow

In [8]:
import mlflow

### Train RF Baseline

In [11]:
import time
import json
import numpy as np
import mlflow
import mlflow.sklearn

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error


# -----------------------------
# MLflow setup
# -----------------------------
mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("ssl_ood_rf_baseline")

# Optional but useful
RUN_GROUP = "rf_baseline_v1"

rf_results = {dataset: [] for dataset in datasets}

for i, dataset in enumerate(datasets):
    with mlflow.start_run(run_name=f"rf_{dataset}"):
        best_mae = float("inf")
        rf_scorer = None
        best_n_trees = None

        x_train = data_dict_np[dataset]["train"]["x"]
        y_train = data_dict_np[dataset]["train"]["y"].ravel()

        x_val = data_dict_np[dataset]["val_id"]["x"]
        y_val = data_dict_np[dataset]["val_id"]["y"].ravel()

        x_test = data_dict_np[dataset]["test"]["x"]
        y_test = data_dict_np[dataset]["test"]["y"].ravel()

        n_trees = 3

        print(f"Predicting for {dataset} using {n_trees} trees")
        start_time = time.time()

        rf = RandomForestRegressor(
            n_estimators=n_trees,
            random_state=42 + i,
            n_jobs=1
        )
        rf.fit(x_train, y_train)

        y_pred_val = rf.predict(x_val)
        mae_val = mean_absolute_error(y_val, y_pred_val)

        if mae_val < best_mae:
            best_mae = mae_val
            rf_scorer = rf
            best_n_trees = n_trees

        y_pred_test = rf_scorer.predict(x_test)
        mae_test = mean_absolute_error(y_test, y_pred_test)
        error_std_test = np.std(np.abs(y_pred_test - y_test))
        error_se_test = error_std_test / np.sqrt(len(y_test))
        runtime_sec = time.time() - start_time

        result = {
            "model": "rf",
            "model_num": i,
            "dataset": dataset,
            "hyperparams": {"n_trees": best_n_trees},
            "n_trees": best_n_trees,
            "val_id_mae": float(best_mae),
            "test_mae": float(mae_test),
            "test_std": float(error_std_test),
            "test_se": float(error_se_test),
        }

        rf_results[dataset].append(result)

        # -----------------------------
        # Log params
        # -----------------------------
        mlflow.log_param("model", "RandomForestRegressor")
        mlflow.log_param("dataset", dataset)
        mlflow.log_param("n_trees", best_n_trees)
        mlflow.log_param("random_state", 42 + i)
        mlflow.log_param("n_jobs", 1)
        mlflow.log_param("run_group", RUN_GROUP)

        # Useful dataset metadata
        mlflow.log_param("n_train", len(x_train))
        mlflow.log_param("n_val_id", len(x_val))
        mlflow.log_param("n_test", len(x_test))
        mlflow.log_param("x_dim", x_train.shape[1])

        # -----------------------------
        # Log metrics
        # -----------------------------
        mlflow.log_metric("val_id_mae", float(best_mae))
        mlflow.log_metric("test_mae", float(mae_test))
        mlflow.log_metric("test_std_abs_error", float(error_std_test))
        mlflow.log_metric("test_se_abs_error", float(error_se_test))
        mlflow.log_metric("runtime_sec", float(runtime_sec))

        # -----------------------------
        # Log artifacts
        # -----------------------------
        # Save result dict as artifact
        artifact_path = f"result_{dataset}.json"
        with open(artifact_path, "w", encoding="utf-8") as f:
            json.dump(result, f, indent=2)
        mlflow.log_artifact(artifact_path)

        # Log model
        mlflow.sklearn.log_model(
            sk_model=rf_scorer,
            artifact_path="model"
        )

        # Optional tags for filtering in UI
        mlflow.set_tag("project", "ssl_for_ood_molecules")
        mlflow.set_tag("dataset", dataset)
        mlflow.set_tag("model_family", "rf")
        mlflow.set_tag("split_eval", "val_id_and_test")

        print(f"--> took {round(runtime_sec, 1)} sec")

2026/04/22 17:51:53 INFO mlflow.tracking.fluent: Experiment with name 'ssl_ood_rf_baseline' does not exist. Creating a new experiment.


Predicting for GFP using 3 trees


2026/04/22 17:52:15 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/22 17:52:15 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


--> took 19.0 sec
🏃 View run rf_GFP at: http://localhost:5000/#/experiments/1/runs/5125e83f90304c51aaf9b61ffeadfe0a
🧪 View experiment at: http://localhost:5000/#/experiments/1
Predicting for AAV using 3 trees


2026/04/22 17:52:25 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/22 17:52:26 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


--> took 0.5 sec
🏃 View run rf_AAV at: http://localhost:5000/#/experiments/1/runs/c75c22eb64914994bece2aaf841b6148
🧪 View experiment at: http://localhost:5000/#/experiments/1
Predicting for TFBind8 using 3 trees


2026/04/22 17:52:32 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/22 17:52:32 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


--> took 0.1 sec
🏃 View run rf_TFBind8 at: http://localhost:5000/#/experiments/1/runs/43812ca45ff946c6ac050227725d0d1c
🧪 View experiment at: http://localhost:5000/#/experiments/1


In [9]:
rf_results = {dataset: [] for dataset in datasets}

for i, dataset in enumerate(datasets):
    
    best_mae = float("inf")
    rf_scorer = None
    best_n_trees = None

    x_train = data_dict_np[dataset]["train"]["x"]
    y_train = data_dict_np[dataset]["train"]["y"].ravel()

    x_val = data_dict_np[dataset]["val_id"]["x"]
    y_val = data_dict_np[dataset]["val_id"]["y"].ravel()

    x_test = data_dict_np[dataset]["test"]["x"]
    y_test = data_dict_np[dataset]["test"]["y"].ravel()

    # If you later want tuning, replace this with a loop over candidate values
    n_trees = 3
    
    print(f'Predicting for {dataset} using {n_trees} trees')
    
    start_time = time.time()
    
    rf = RandomForestRegressor(
        n_estimators=n_trees,
        random_state=42 + i,
        n_jobs=1
    )
    rf.fit(x_train, y_train)
    
    y_pred_val = rf.predict(x_val)
    mae_val = mean_absolute_error(y_val, y_pred_val)
    
    if mae_val < best_mae:
        best_mae = mae_val
        rf_scorer = rf
        best_n_trees = n_trees
    
    # Test evaluation
    y_pred_test = rf_scorer.predict(x_test)
    mae_test = mean_absolute_error(y_test, y_pred_test)
    error_std_test = np.std(np.abs(y_pred_test - y_test))
    error_se_test = error_std_test / np.sqrt(len(y_test))
    
    result = {
        "model": "rf",
        "model_num": i,
        "dataset": dataset,
        "hyperparams": {"n_trees": best_n_trees},
        "n_trees": best_n_trees,
        "val_id_mae": float(best_mae),
        "test_mae": float(mae_test),
        "test_std": float(error_std_test),
        "test_se": float(error_se_test),
    }
    
    rf_results[dataset].append(result)
    
    print(f'--> took {round(time.time()-start_time, 1)} sec')

Predicting for GFP using 3 trees
--> took 18.2 sec
Predicting for AAV using 3 trees
--> took 0.5 sec
Predicting for TFBind8 using 3 trees
--> took 0.1 sec


Predicting for GFP using 1111 trees
--> took 7398.9 sec
Predicting for AAV using 1111 trees
--> took 2.1 sec

Predicting for GFP using 11 trees
--> took 369.1 sec
Predicting for AAV using 11 trees
--> took 9.9 sec
Predicting for TFBind8 using 11 trees
--> took 0.4 sec

In [10]:
pd.concat([pd.DataFrame(rf_results[dataset]) for dataset in datasets]).reset_index(drop=True)

,model,model_num,dataset,hyperparams,n_trees,val_id_mae,test_mae,test_std,test_se
0,rf,0,GFP,{'n_trees': 3},3,0.279051,0.669729,0.723311,0.007498
1,rf,1,AAV,{'n_trees': 3},3,0.447648,0.821379,0.613730,0.008804
2,rf,2,TFBind8,{'n_trees': 3},3,0.562154,0.717644,0.566536,0.003644


In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.append(str(PROJECT_ROOT / "src"))

from ssl_for_ood.training.trainer import load_config, train_single_run

config = load_config(
    PROJECT_ROOT / "config" / "lstm" / "base.yaml",
    PROJECT_ROOT / "config" / "lstm" / "aav.yaml",
)

config["output"]["run_name"] = "lstm_aav_manual_run"
config["mlflow"]["enabled"] = True   # or False
config["training"]["enable_progress_bar"] = True
config["training"]["epochs"] = 5

metrics, artifacts = train_single_run(config)

Experiment with name ssl-ood-lstm not found. Creating it.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name    ┃ Type          ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model   │ LSTMRegressor │  155 K │ train │     0 │
│ 1 │ loss_fn │ MSELoss       │      0 │ train │     0 │
└───┴─────────┴───────────────┴────────┴───────┴───────┘

Trainable params: 155 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 155 K                                                                                                
Total estimated model params size (MB): 0                                                                          
Modules in train mode: 9                                                                                           
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

C:\Users\kroep\anaconda3\envs\ssl_ood_env\Lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:
434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of 
the `num_workers` argument` to `num_workers=5` in the `DataLoader` to improve performance.

C:\Users\kroep\anaconda3\envs\ssl_ood_env\Lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:
434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of 
the `num_workers` argument` to `num_workers=5` in the `DataLoader` to improve performance.

`Trainer.fit` stopped: `max_epochs=5` reached.


🏃 View run lstm__manual at: http://127.0.0.1:5000/#/experiments/3/runs/d336604892be4dffa7268ed180867d70
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3


C:\Users\kroep\anaconda3\envs\ssl_ood_env\Lib\site-packages\lightning_fabric\utilities\cloud_io.py:73: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.


In [1]:
from pathlib import Path
import sys
import time

PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.append(str(PROJECT_ROOT / "src"))

from ssl_for_ood.training.trainer import load_config, train_single_run

# model_class = "lstm"

models_with_errors = ['mlp']
models_working = ['cnn', 'lstm', 'cnn_lstm', 'lstm_cnn', 'transformer']

for model_class in models_with_errors:
    
    print('##################')
    print(f'#####  {model_class.upper()}')
    print('##################')
    
    for d in ['tfbind8', 'aav', 'gfp']:

        print(f'\n### Dataset: {d}\n')

        config = load_config(
            PROJECT_ROOT / "config" / model_class / "base.yaml",
            PROJECT_ROOT / "config" / model_class / f"{d}.yaml",
        )

        config["output"]["run_name"] = f"{model_class}_{d}_manual_run"
        config["mlflow"]["enabled"] = False
        config["training"]["enable_progress_bar"] = True
        config["training"]["epochs"] = 3

        start_time = time.time()
        metrics, artifacts = train_single_run(config)
        print(f'--> took {round(time.time()-start_time, 2)} sec')

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
C:\Users\kroep\anaconda3\envs\ssl_ood_env\Lib\site-packages\pytorch_lightning\trainer\connectors\logger_connector\logger_connector.py:76: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of the `pytorch_lightning` package, due to potential conflicts with other packages in the ML ecosystem. For this reason, `logger=True` will use `CSVLogger` as the default logger, unless the `tensorboard` or `tensorboardX` packages are found. Please `pip install lightning[extra]` or one of them to enable TensorBoard support by default
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


##################
#####  MLP
##################

### Dataset: tfbind8



┏━━━┳━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name    ┃ Type         ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model   │ MLPRegressor │  140 K │ train │     0 │
│ 1 │ loss_fn │ MSELoss      │      0 │ train │     0 │
└───┴─────────┴──────────────┴────────┴───────┴───────┘

Trainable params: 140 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 140 K                                                                                                
Total estimated model params size (MB): 0                                                                          
Modules in train mode: 14                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

...

┏━━━┳━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name    ┃ Type         ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model   │ MLPRegressor │  1.4 M │ train │     0 │
│ 1 │ loss_fn │ MSELoss      │      0 │ train │     0 │
└───┴─────────┴──────────────┴────────┴───────┴───────┘

Trainable params: 1.4 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 1.4 M                                                                                                
Total estimated model params size (MB): 5                                                                          
Modules in train mode: 14                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

`Trainer.fit` stopped: `max_epochs=3` reached.


--> took 23.14 sec


#### One more improvement idea:
- after every run, save that to the results csv file, not just at te end

### Hyperparameter Search

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()

if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.append(str(PROJECT_ROOT / "src"))

from ssl_for_ood.training.orchestrator import run_experiments

df = run_experiments(
    project_root=PROJECT_ROOT,
    model_names=["mlp", 'cnn', 'lstm', 'cnn_lstm', 'lstm_cnn', 'transformer'],
    dataset_names=["aav"],
    n_trials=3,
    use_mlflow=True,
)


🚀 Running mlp on aav
[mlp | aav] Trial 1/3


Experiment with name ssl-ood-mlp not found. Creating it.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
C:\Users\kroep\anaconda3\envs\ssl_ood_env\Lib\site-packages\pytorch_lightning\callbacks\model_checkpoint.py:881: Checkpoint directory C:\Users\kroep\Desktop\AI\ssl-for-ood-molecules\results\training\mlp\aav\mlp_aav_trial_0\checkpoints exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name    ┃ Type         ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model   │ MLPRegressor │  550 K │ train │     0 │
│ 1 │ loss_fn │ MSELoss      │      0 │ train │     0 │
└───┴─────────┴──────────────┴────────┴───────┴───────┘

Trainable params: 550 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 550 K                                                                                                
Total estimated model params size (MB): 2                                                                          
Modules in train mode: 11                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

...

[transformer | aav] Trial 3/3


┏━━━┳━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name    ┃ Type                 ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model   │ TransformerRegressor │  115 K │ train │     0 │
│ 1 │ loss_fn │ MSELoss              │      0 │ train │     0 │
└───┴─────────┴──────────────────────┴────────┴───────┴───────┘

Trainable params: 115 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 115 K                                                                                                
Total estimated model params size (MB): 0                                                                          
Modules in train mode: 41                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

🏃 View run transformer__aav__trial_2 at: http://127.0.0.1:5000/#/experiments/7/runs/4dbf2f78cb434f7d86b032c8b3b19e14
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/7

✅ Saved results to C:\Users\kroep\Desktop\AI\ssl-for-ood-molecules\results\hyperparameter_search_20260426_045355.csv


In [11]:
import pandas as pd
# pd.read_csv('C:\\Users\\kroep\\Desktop\\AI\\ssl-for-ood-molecules\\results\\hyperparameter_search_20260424_175625.csv')
pd.read_csv('C:\\Users\\kroep\\Desktop\\AI\\ssl-for-ood-molecules\\results\\hyperparameter_search_20260426_045355.csv')

,model,dataset,trial,duration_sec,config,val_id_mae,val_ood_mae,val_id_std,val_ood_std,selected_metric,best_epoch,test_mae,run_dir
0,mlp,aav,0,289.786192,"{""training"": {""batch_size"": 128, ""learning_rat...",0.978137,1.092521,0.849153,0.952246,1.092521,NaN,NaN,results\training\mlp\aav\mlp_aav_trial_0
1,mlp,aav,1,639.084651,"{""training"": {""batch_size"": 32, ""learning_rate...",0.841638,0.860719,0.766619,0.798411,0.860719,NaN,NaN,results\training\mlp\aav\mlp_aav_trial_1
2,mlp,aav,2,1631.008697,"{""training"": {""batch_size"": 32, ""learning_rate...",0.858299,0.973483,0.779051,0.851956,0.973483,NaN,NaN,results\training\mlp\aav\mlp_aav_trial_2
3,cnn,aav,0,528.451471,"{""training"": {""batch_size"": 64, ""learning_rate...",1.474655,1.904841,1.182110,1.487413,1.904841,NaN,NaN,results\training\cnn\aav\cnn_aav_trial_0
4,cnn,aav,1,670.250798,"{""training"": {""batch_size"": 128, ""learning_rat...",0.894690,1.029059,0.798634,0.881874,1.029059,NaN,NaN,results\training\cnn\aav\cnn_aav_trial_1
5,cnn,aav,2,2172.024874,"{""training"": {""batch_size"": 32, ""learning_rate...",1.238528,1.580097,1.050812,1.307495,1.580097,NaN,NaN,results\training\cnn\aav\cnn_aav_trial_2
6,lstm,aav,0,3127.230789,"{""training"": {""batch_size"": 64, ""learning_rate...",0.886277,0.990651,0.812271,0.867238,0.990651,NaN,NaN,results\training\lstm\aav\lstm_aav_trial_0
7,lstm,aav,1,815.381449,"{""training"": {""batch_size"": 128, ""learning_rat...",0.806358,0.853500,0.742979,0.793449,0.853500,NaN,NaN,results\training\lstm\aav\lstm_aav_trial_1
8,lstm,aav,2,1666.285206,"{""training"": {""batch_size"": 32, ""learning_rate...",1.056774,1.254962,0.969375,1.142434,1.254962,NaN,NaN,results\training\lstm\aav\lstm_aav_trial_2
9,cnn_lstm,aav,0,1363.942436,"{""training"": {""batch_size"": 64, ""learning_rate...",0.935697,1.104485,0.861588,1.004989,1.104485,NaN,NaN,results\training\cnn_lstm\aav\cnn_lstm_aav_tri...


In [3]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()

if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.append(str(PROJECT_ROOT / "src"))

from ssl_for_ood.training.normalization_orchestrator import run_normalization_experiments
from ssl_for_ood.training.trainer import load_config, train_single_run

'''
df = run_normalization_experiments(
    project_root=PROJECT_ROOT,
    model_names=["cnn_lstm"], #["mlp", "cnn", "lstm", "cnn_lstm", "lstm_cnn", "transformer"],
    dataset_names=["aav"], #["tfbind8", "gfp", "aav"],
    strategies=[
        #None,
        #"architecture_native_norm",
        "input_bn",
        "final_bn",
    ],
    use_mlflow=False,
)
'''

'\ndf = run_normalization_experiments(\n    project_root=PROJECT_ROOT,\n    model_names=["cnn_lstm"], #["mlp", "cnn", "lstm", "cnn_lstm", "lstm_cnn", "transformer"],\n    dataset_names=["aav"], #["tfbind8", "gfp", "aav"],\n    strategies=[\n        #None,\n        #"architecture_native_norm",\n        "input_bn",\n        "final_bn",\n    ],\n    use_mlflow=False,\n)\n'

In [10]:
for model_name in ["mlp", 'cnn', 'lstm', 'cnn_lstm', 'lstm_cnn', 'transformer']:
    
    print(f'''
###############
### {model_name}
###############
''')
    
    config = load_config(
        PROJECT_ROOT / "config" / model_name / "base.yaml",
        PROJECT_ROOT / "config" / model_name / "aav.yaml",
    )
    
    for strategy in ["architecture_native_norm", "input_bn", "final_bn", None]:
        
        print(f'\n   --> STRATEGY: "{strategy}"')
        
        config["model"]["normalization_strategy"] = strategy
        config["dataset"]["name"] = "aav"
        config["training"]["epochs"] = 3

        metrics, artifacts = train_single_run(config)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]



###############
### mlp
###############


   --> STRATEGY: "architecture_native_norm"


┏━━━┳━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name    ┃ Type         ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model   │ MLPRegressor │  276 K │ train │     0 │
│ 1 │ loss_fn │ MSELoss      │      0 │ train │     0 │
└───┴─────────┴──────────────┴────────┴───────┴───────┘

Trainable params: 276 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 276 K                                                                                                
Total estimated model params size (MB): 1                                                                          
Modules in train mode: 19                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

...

┏━━━┳━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name    ┃ Type                 ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model   │ TransformerRegressor │  105 K │ train │     0 │
│ 1 │ loss_fn │ MSELoss              │      0 │ train │     0 │
└───┴─────────┴──────────────────────┴────────┴───────┴───────┘

Trainable params: 105 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 105 K                                                                                                
Total estimated model params size (MB): 0                                                                          
Modules in train mode: 33                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

`Trainer.fit` stopped: `max_epochs=3` reached.


🏃 View run transformer__manual at: http://127.0.0.1:5000/#/experiments/7/runs/f0b3c8c20b104b1bba03c2f167fa90d8
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/7


### Experiment 1: BN and LN for best hyperparameter settings

In [1]:
from pathlib import Path
import sys
import json
import ast
import pandas as pd

PROJECT_ROOT = Path.cwd()

if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.append(str(PROJECT_ROOT / "src"))

# from ssl_for_ood.training.trainer import load_config, train_single_run
from ssl_for_ood.training.normalization_orchestrator import run_normalization_from_best_configs

df_norm = run_normalization_from_best_configs(
    project_root=PROJECT_ROOT,
    search_results_path=PROJECT_ROOT / "results" / "hyperparameter_search_20260426_045355.csv",
    dataset_name="aav",
)


###############
### cnn
###############

   --> STRATEGY: "architecture_native_norm"


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name    ┃ Type         ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model   │ CNNRegressor │ 56.3 K │ train │     0 │
│ 1 │ loss_fn │ MSELoss      │      0 │ train │     0 │
└───┴─────────┴──────────────┴────────┴───────┴───────┘

Trainable params: 56.3 K                                                                                           
Non-trainable params: 0                                                                                            
Total params: 56.3 K                                                                                               
Total estimated model params size (MB): 0                                                                          
Modules in train mode: 19                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

...

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name    ┃ Type                 ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model   │ TransformerRegressor │  213 K │ train │     0 │
│ 1 │ loss_fn │ MSELoss              │      0 │ train │     0 │
└───┴─────────┴──────────────────────┴────────┴───────┴───────┘

Trainable params: 213 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 213 K                                                                                                
Total estimated model params size (MB): 0                                                                          
Modules in train mode: 33                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

🏃 View run transformer__manual at: http://127.0.0.1:5000/#/experiments/7/runs/42cfae9e62c04007b204720197734170
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/7

✅ Saved normalization results to C:\Users\kroep\Desktop\AI\ssl-for-ood-molecules\results\normalization_from_best_aav_20260430_003028.csv


In [1]:
import pandas as pd
df_norm = pd.read_csv('C:\\Users\\kroep\\Desktop\\AI\\ssl-for-ood-molecules\\results\\normalization_from_best_aav_20260430_003028.csv')

In [7]:
df_norm.sort_values('val_ood_mae')

,model,dataset,normalization_strategy,duration_sec,val_id_mae,val_ood_mae,test_mae,val_id_std_abs_error,val_ood_std_abs_error,test_std_abs_error,run_dir
10,lstm,aav,final_bn,847.349832,0.793860,0.807032,NaN,0.743731,0.738349,NaN,results\training\lstm\aav\20260429_200426
9,lstm,aav,input_bn,543.227698,0.788206,0.829678,NaN,0.740845,0.773375,NaN,results\training\lstm\aav\20260429_195523
11,lstm,aav,NaN,643.805017,0.797056,0.847245,NaN,0.742440,0.783192,NaN,results\training\lstm\aav\20260429_201833
13,lstm_cnn,aav,input_bn,2119.971410,0.814337,0.855202,NaN,0.772425,0.774230,NaN,results\training\lstm_cnn\aav\20260429_205827
2,cnn,aav,final_bn,719.427601,0.800898,0.869613,NaN,0.725678,0.775935,NaN,results\training\cnn\aav\20260429_181215
8,lstm,aav,architecture_native_norm,730.775943,0.797066,0.871531,NaN,0.732724,0.803768,NaN,results\training\lstm\aav\20260429_194312
16,mlp,aav,architecture_native_norm,1384.428490,0.815231,0.879079,NaN,0.740319,0.773760,NaN,results\training\mlp\aav\20260429_224428
19,mlp,aav,NaN,605.092372,0.836381,0.895930,NaN,0.772084,0.795483,NaN,results\training\mlp\aav\20260429_233025
1,cnn,aav,input_bn,545.378868,0.830036,0.901543,NaN,0.754924,0.782579,NaN,results\training\cnn\aav\20260429_180310
18,mlp,aav,final_bn,941.330916,0.835502,0.918007,NaN,0.758974,0.825776,NaN,results\training\mlp\aav\20260429_231443


In [6]:
df_norm.iloc[int(df_norm.val_ood_mae.argmin())]

model                                                          lstm
dataset                                                         aav
normalization_strategy                                     final_bn
duration_sec                                             847.349832
val_id_mae                                                  0.79386
val_ood_mae                                                0.807032
test_mae                                                        NaN
val_id_std_abs_error                                       0.743731
val_ood_std_abs_error                                      0.738349
test_std_abs_error                                              NaN
run_dir                   results\training\lstm\aav\20260429_200426
Name: 10, dtype: object

In [2]:
for model in df_norm.model.unique():
    for normalization_strategy in df_norm.normalization_strategy.unique():
        

,model,dataset,normalization_strategy,duration_sec,val_id_mae,val_ood_mae,test_mae,val_id_std_abs_error,val_ood_std_abs_error,test_std_abs_error,run_dir
0,cnn,aav,architecture_native_norm,255.602809,0.844793,0.921726,NaN,0.771924,0.793598,NaN,results\training\cnn\aav\20260429_175854
1,cnn,aav,input_bn,545.378868,0.830036,0.901543,NaN,0.754924,0.782579,NaN,results\training\cnn\aav\20260429_180310
2,cnn,aav,final_bn,719.427601,0.800898,0.869613,NaN,0.725678,0.775935,NaN,results\training\cnn\aav\20260429_181215
3,cnn,aav,NaN,529.911274,0.865184,0.962309,NaN,0.779956,0.829530,NaN,results\training\cnn\aav\20260429_182414
4,cnn_lstm,aav,architecture_native_norm,831.083285,1.019122,1.391930,NaN,0.927741,1.248771,NaN,results\training\cnn_lstm\aav\20260429_183304
5,cnn_lstm,aav,input_bn,1032.831557,0.949807,1.158798,NaN,0.864152,1.037596,NaN,results\training\cnn_lstm\aav\20260429_184655
6,cnn_lstm,aav,final_bn,975.378326,1.016772,1.330689,NaN,0.914128,1.169556,NaN,results\training\cnn_lstm\aav\20260429_190408
7,cnn_lstm,aav,NaN,1368.064842,0.935697,1.104485,NaN,0.861588,1.004989,NaN,results\training\cnn_lstm\aav\20260429_192024
8,lstm,aav,architecture_native_norm,730.775943,0.797066,0.871531,NaN,0.732724,0.803768,NaN,results\training\lstm\aav\20260429_194312
9,lstm,aav,input_bn,543.227698,0.788206,0.829678,NaN,0.740845,0.773375,NaN,results\training\lstm\aav\20260429_195523
